In [ ]:
import cv2
import matplotlib.pyplot as plt

# Cargar imágenes estéreo (ejemplo sintético)
left = cv2.imread(cv2.samples.findFile('samples/data/aloeL.jpg'), cv2.IMREAD_GRAYSCALE)
right = cv2.imread(cv2.samples.findFile('samples/data/aloeR.jpg'), cv2.IMREAD_GRAYSCALE)

stereo = cv2.StereoBM_create(numDisparities=16*5, blockSize=15)
disparity = stereo.compute(left, right)

plt.imshow(disparity, 'gray')
plt.title("Mapa de disparidad")
plt.show()


In [ ]:
import numpy as np
import open3d as o3d

h, w = left.shape
focal_length = 0.8*w
Q = np.float32([[1, 0, 0, -w/2],
                [0,-1,0,h/2],
                [0,0,0,-focal_length],
                [0,0,1,0]])

points_3D = cv2.reprojectImageTo3D(disparity, Q)
mask = disparity > disparity.min()
out_points = points_3D[mask]

# Crear nube de puntos
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(out_points)

o3d.visualization.draw_geometries([pcd])


In [ ]:
import cv2

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Dibujar un cuadrado virtual en el centro de la imagen
    h, w, _ = frame.shape
    cv2.rectangle(frame, (w//2-50, h//2-50), (w//2+50, h//2+50), (0,255,0), 3)
    cv2.putText(frame, 'Objeto Virtual', (w//2-60, h//2-60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    cv2.imshow("Realidad Aumentada Basica", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands()
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)

    cv2.imshow("RA - Deteccion de Manos", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
